# Volume 19 corrigido — vetores E5 no Google Colab

Este notebook gera um corpus **novo e separado** a partir da versão conferida do
volume 19. O texto das páginas PDF **10–232**, os IDs, as referências e os offsets
acompanham o pacote. Os vetores antigos não são reutilizados.

1. Escolha **Ambiente de execução → Alterar tipo de ambiente de execução → GPU**
   se houver essa opção. A execução em CPU também é possível, mas costuma demorar.
2. Execute as células em ordem, ou use **Ambiente de execução → Executar tudo**.
3. Ao terminar, salve o ZIP gerado e o relatório JSON. Envie o relatório para
   conferirmos a execução; o ZIP será usado na integração no Mac.

Ao executar, o texto do livro e seus registros serão processados no **Google
Colab**. O modelo será baixado explicitamente do Hugging Face, na revisão fixa
indicada abaixo. O notebook não altera sua instalação local nem ativa a busca.
Evite encerrar a sessão antes de baixar os resultados: os arquivos em `/content`
são temporários. A geração não substitui a conferência editorial do livro.


## Configuração

O lote inicial é 2 para reduzir o consumo de memória. Você pode reduzi-lo para 1.
`auto` escolhe GPU CUDA disponível ou CPU. Para exigir GPU, selecione `cuda`;
nessa opção, uma sessão sem GPU para com uma mensagem clara.

O modelo é `intfloat/multilingual-e5-large`, revisão
`3d7cfbdacd47fdda877c5cd8a79fbcc4f2a574f3`: inferência em **float32**,
prefixo `passage: `, pooling médio e normalização L2. Os fragmentos respeitam o
limite real do tokenizador; não há truncamento de conteúdo.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import uuid

LOTE = 2
DISPOSITIVO = "auto"  # "auto", "cuda" ou "cpu"

# A versão e a soma abaixo são fixadas na publicação deste notebook.
URL_PACOTE = "https://raw.githubusercontent.com/andregirogiro-pixel/lingua/6f7405787b34467c2af7718dc6454577d0db3aa5/busca/etapa3_busca.zip"
SHA256_PACOTE = "605e807571cc1778aaaead56903fa8301b3134a90e3f3f463ae64dd69321289e"
MODELO = "intfloat/multilingual-e5-large"
REVISAO_MODELO = "3d7cfbdacd47fdda877c5cd8a79fbcc4f2a574f3"
FONTE_PDF_SHA256 = "1b4ea213276f48f77b53349109c9d9a352ade2225de43aa50765225db5c0a3ba"

if not isinstance(LOTE, int) or isinstance(LOTE, bool) or not 1 <= LOTE <= 64:
    raise ValueError("LOTE deve ser um inteiro entre 1 e 64.")
if DISPOSITIVO not in {"auto", "cuda", "cpu"}:
    raise ValueError("DISPOSITIVO deve ser auto, cuda ou cpu.")

# Reexecutar esta célula mantém a mesma execução. A geração recusa sobrescrita.
if "EXECUCAO_ID" not in globals():
    EXECUCAO_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
PASTA_EXECUCAO = Path("/content/volume19_execucoes") / EXECUCAO_ID
PASTA_EXECUCAO.mkdir(parents=True, exist_ok=True)
PASTA_CACHE = Path("/content/cache_e5_volume19")
PASTA_SAIDA = PASTA_EXECUCAO / "volume19_camada_a_novo"
print("Execução:", EXECUCAO_ID)
print("Saída nova:", PASTA_SAIDA)
print("Modelo:", MODELO, "— revisão", REVISAO_MODELO)


## Preparar o ambiente do Colab

A célula usa o Python desta sessão e o PyTorch que já vem com o Colab.
Instala as bibliotecas do tokenizador/modelo em versões compatíveis; não depende
do ambiente E5 do seu Mac. Execute em uma sessão nova do Colab.


In [ ]:
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    raise RuntimeError("PyTorch ausente. Abra este notebook em um ambiente Python do Google Colab e tente novamente.")

# O processamento usa apenas PyTorch; não carrega TensorFlow nem Flax.
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
sys.dont_write_bytecode = True
subprocess.run([
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "numpy>=1.26,<3", "transformers==4.51.3", "huggingface-hub>=0.30,<1",
    "safetensors>=0.4,<1", "sentencepiece>=0.2,<1",
], check=True)

import numpy as np
import torch
import transformers
import importlib.metadata as metadata
from transformers import AutoModel, AutoTokenizer

if transformers.__version__ != "4.51.3":
    raise RuntimeError("A sessão já carregou outra versão do Transformers. Reinicie o ambiente do Colab e execute o notebook desde o início.")
if DISPOSITIVO == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("GPU CUDA indisponível. Selecione GPU nas configurações do Colab ou escolha DISPOSITIVO='cpu'.")
DISPOSITIVO_REAL = ("cuda" if torch.cuda.is_available() else "cpu") if DISPOSITIVO == "auto" else DISPOSITIVO
VERSOES_SESSAO = {nome: metadata.version(nome) for nome in (
    "numpy", "torch", "transformers", "huggingface-hub", "safetensors", "sentencepiece"
)}
print("Python:", sys.version.split()[0])
print("Bibliotecas:", VERSOES_SESSAO)
print("Dispositivo:", DISPOSITIVO_REAL)
if DISPOSITIVO_REAL == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Execução em CPU selecionada; a geração pode demorar mais.")


## Conferir e abrir o pacote

O download contém o preparador e o texto já extraído, com seus mapas e hashes.
O PDF não é necessário nesta execução. A soma SHA-256 fixa garante que o Colab
utilize exatamente o pacote publicado com este notebook.


In [ ]:
import hashlib
import json
import re
import stat
import urllib.request
import zipfile
from pathlib import PurePosixPath

def sha256_arquivo(caminho):
    resumo = hashlib.sha256()
    with Path(caminho).open("rb") as arquivo:
        for trecho in iter(lambda: arquivo.read(1024 * 1024), b""):
            resumo.update(trecho)
    return resumo.hexdigest()

if not re.fullmatch(r"[0-9a-f]{64}", SHA256_PACOTE):
    raise RuntimeError("Este notebook não foi finalizado: falta a soma fixa do pacote. Use a versão publicada no GitHub.")

ARQUIVO_PACOTE = PASTA_EXECUCAO / "etapa3_busca.zip"
if not ARQUIVO_PACOTE.exists():
    arquivo_temporario = ARQUIVO_PACOTE.with_suffix(".download")
    if arquivo_temporario.exists():
        raise RuntimeError("Há um download parcial nesta execução. Abra uma nova sessão para tentar novamente.")
    with urllib.request.urlopen(URL_PACOTE, timeout=120) as resposta, arquivo_temporario.open("xb") as arquivo:
        total = 0
        while True:
            trecho = resposta.read(1024 * 1024)
            if not trecho:
                break
            total += len(trecho)
            if total > 100 * 1024 * 1024:
                raise RuntimeError("O pacote excede o tamanho esperado.")
            arquivo.write(trecho)
    if sha256_arquivo(arquivo_temporario) != SHA256_PACOTE:
        raise RuntimeError("O SHA-256 do download diverge. A execução foi interrompida sem importar código.")
    arquivo_temporario.rename(ARQUIVO_PACOTE)
if sha256_arquivo(ARQUIVO_PACOTE) != SHA256_PACOTE:
    raise RuntimeError("O arquivo local diverge da versão publicada; não será reutilizado.")

PASTA_PACOTE = PASTA_EXECUCAO / "pacote"
with zipfile.ZipFile(ARQUIVO_PACOTE) as pacote:
    entradas = pacote.infolist()
    nomes = set()
    bytes_descompactados = 0
    for entrada in entradas:
        partes = PurePosixPath(entrada.filename).parts
        if (not partes or entrada.filename.startswith("/") or "\\" in entrada.filename
                or "\x00" in entrada.filename or any(parte in {".", ".."} for parte in partes)
                or entrada.filename in nomes or stat.S_ISLNK(entrada.external_attr >> 16)):
            raise RuntimeError("O ZIP contém um caminho inválido ou repetido.")
        nomes.add(entrada.filename)
        bytes_descompactados += entrada.file_size
    if bytes_descompactados > 100 * 1024 * 1024:
        raise RuntimeError("O conteúdo descompactado excede o tamanho esperado.")
    # A repetição confere os bytes existentes, em vez de sobrescrever arquivos.
    for entrada in entradas:
        destino = PASTA_PACOTE.joinpath(*PurePosixPath(entrada.filename).parts)
        if not destino.resolve().is_relative_to(PASTA_PACOTE.resolve()):
            raise RuntimeError("Caminho fora da pasta de extração.")
        if entrada.is_dir():
            destino.mkdir(parents=True, exist_ok=True)
            continue
        conteudo = pacote.read(entrada)
        if destino.exists():
            if destino.is_symlink() or not destino.is_file() or destino.read_bytes() != conteudo:
                raise RuntimeError("Um arquivo extraído foi alterado; a execução não reutiliza esse conteúdo.")
        else:
            destino.parent.mkdir(parents=True, exist_ok=True)
            with destino.open("xb") as arquivo:
                arquivo.write(conteudo)

RAIZ_PACOTE = PASTA_PACOTE / "etapa3_busca"
ARQUIVO_PREPARADOR = RAIZ_PACOTE / "arquivos/agente_analista/preparar_volume19.py"
PASTA_FONTE = RAIZ_PACOTE / "volume19_fonte_v2"
if not ARQUIVO_PREPARADOR.is_file() or not (PASTA_FONTE / "manifesto.json").is_file():
    raise RuntimeError("O pacote não contém o preparador e a fonte esperados.")
print("Pacote conferido:", SHA256_PACOTE)
print("Texto e mapas:", PASTA_FONTE)


## Gerar os vetores novos

Nesta célula ocorre o download explícito do E5 e a inferência. O preparador
valida os hashes da fonte antes de gerar qualquer fragmento. Cada entrada exata
de vetorização e sua posição na matriz são registradas no corpus novo.

A saída deve ser uma pasta nova. Se a execução parar ou você tentar repetir esta
célula, o código recusa sobrescrever resultados. Para outra execução, abra uma
nova sessão do Colab e execute novamente; o resultado anterior deve ser baixado
antes de encerrar sua sessão.


In [ ]:
import importlib.util

nome_modulo = "preparador_volume19_colab"
especificacao = importlib.util.spec_from_file_location(nome_modulo, ARQUIVO_PREPARADOR)
if especificacao is None or especificacao.loader is None:
    raise RuntimeError("Não foi possível carregar o preparador conferido.")
preparador = importlib.util.module_from_spec(especificacao)
sys.modules[nome_modulo] = preparador
especificacao.loader.exec_module(preparador)
if (preparador.MODELO != MODELO or preparador.REVISAO != REVISAO_MODELO
        or preparador.SHA_PDF != FONTE_PDF_SHA256):
    raise RuntimeError("O preparador não corresponde ao modelo, revisão e fonte fixados neste notebook.")

if PASTA_SAIDA.exists():
    raise RuntimeError("A pasta desta geração já existe. Nenhum resultado será sobrescrito. Baixe os arquivos concluídos nas próximas células, ou abra uma nova sessão para gerar outra execução.")

def mostrar_progresso(*args, **kwargs):
    print("Progresso:", *args, kwargs if kwargs else "", flush=True)

RELATORIO_GERACAO = preparador.preparar_volume19(
    PASTA_FONTE, PASTA_SAIDA,
    cache_modelos=PASTA_CACHE,
    baixar_modelo=True,
    dispositivo=DISPOSITIVO_REAL,
    lote=LOTE,
    progresso=mostrar_progresso,
)
print(json.dumps(RELATORIO_GERACAO, ensure_ascii=False, indent=2))


## Conferir integralmente antes de baixar

O verificador relê os arquivos, offsets, entradas, IDs, linhas de vetores e
normas. Só depois dessa conferência o notebook monta o ZIP. Uma execução de
teste ou um conjunto de vetores incompleto não é aceito como resultado real.

O arquivo resultante continua **separado da busca ativa**. A integração e os
testes no Mac ocorrerão depois que conferirmos este relatório.


In [ ]:
RELATORIO_VALIDACAO = preparador.validar_corpus_volume19(PASTA_SAIDA)
MANIFESTO_SAIDA = json.loads((PASTA_SAIDA / "manifesto.json").read_text(encoding="utf-8"))
RELATORIO_GERACAO = json.loads((PASTA_SAIDA / "geracao.json").read_text(encoding="utf-8"))

# Estas verificações são adicionais à leitura integral feita pelo preparador.
if (RELATORIO_GERACAO.get("corpus_validado") is not True
        or RELATORIO_VALIDACAO.get("corpus_validado") is not True):
    raise RuntimeError("A geração ou a releitura integral não confirmou um corpus válido.")
if (RELATORIO_GERACAO.get("modelo") != MODELO
        or RELATORIO_GERACAO.get("revisao") != REVISAO_MODELO
        or RELATORIO_GERACAO.get("sha256_pdf") != FONTE_PDF_SHA256):
    raise RuntimeError("O relatório diverge do modelo, revisão ou PDF de origem fixados.")
if MANIFESTO_SAIDA.get("embedding_real") is not True:
    raise RuntimeError("O manifesto não confirma embeddings reais; o notebook não publicará essa saída.")
if MANIFESTO_SAIDA.get("pronto_para_busca") is not False:
    raise RuntimeError("A saída deve permanecer separada da ativação da busca nesta etapa.")
print(json.dumps(RELATORIO_VALIDACAO, ensure_ascii=False, indent=2))
print("Fonte, corpus e vetores relidos. A ativação no aplicativo continua pendente.")


In [ ]:
NOME_RESULTADO = "volume19_vetores_colab_" + EXECUCAO_ID
ARQUIVO_RESULTADO = PASTA_EXECUCAO / (NOME_RESULTADO + ".zip")
ARQUIVO_RELATORIO = PASTA_EXECUCAO / (NOME_RESULTADO + ".json")

if ARQUIVO_RESULTADO.exists() or ARQUIVO_RELATORIO.exists():
    raise RuntimeError("Este resultado já foi empacotado. Use a célula de download abaixo; os arquivos não serão sobrescritos.")

arquivos_corpus = sorted(caminho for caminho in PASTA_SAIDA.rglob("*") if caminho.is_file())
if not arquivos_corpus:
    raise RuntimeError("A geração não produziu arquivos para empacotar.")
HASHES_RESULTADO = {
    caminho.relative_to(PASTA_SAIDA).as_posix(): {
        "sha256": sha256_arquivo(caminho), "bytes": caminho.stat().st_size
    }
    for caminho in arquivos_corpus
}
with zipfile.ZipFile(ARQUIVO_RESULTADO, "x", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as pacote:
    for caminho in arquivos_corpus:
        if caminho.is_symlink() or not caminho.resolve().is_relative_to(PASTA_SAIDA.resolve()):
            raise RuntimeError("Arquivo de saída fora do corpus; o ZIP foi interrompido.")
        pacote.write(caminho, (Path("volume19_camada_a_novo") / caminho.relative_to(PASTA_SAIDA)).as_posix())
with zipfile.ZipFile(ARQUIVO_RESULTADO) as pacote:
    arquivo_com_erro = pacote.testzip()
    if arquivo_com_erro is not None:
        raise RuntimeError("O ZIP de resultados falhou na conferência: " + arquivo_com_erro)
    for nome, registro in HASHES_RESULTADO.items():
        conteudo = pacote.read("volume19_camada_a_novo/" + nome)
        if len(conteudo) != registro["bytes"] or hashlib.sha256(conteudo).hexdigest() != registro["sha256"]:
            raise RuntimeError("Um arquivo do ZIP diverge do corpus validado: " + nome)

RELATORIO_COLAB = {
    "etapa": "volume19_vetores_colab",
    "execucao_id": EXECUCAO_ID,
    "data_utc": datetime.now(timezone.utc).isoformat(),
    "pacote_codigo_sha256": SHA256_PACOTE,
    "modelo": MODELO,
    "revisao_modelo": REVISAO_MODELO,
    "fonte_pdf_sha256": FONTE_PDF_SHA256,
    "dispositivo": DISPOSITIVO_REAL,
    "lote": LOTE,
    "python": sys.version.split()[0],
    "bibliotecas": VERSOES_SESSAO,
    "geracao": RELATORIO_GERACAO,
    "validacao_integral": RELATORIO_VALIDACAO,
    "arquivo_resultado": ARQUIVO_RESULTADO.name,
    "arquivo_resultado_bytes": ARQUIVO_RESULTADO.stat().st_size,
    "arquivo_resultado_sha256": sha256_arquivo(ARQUIVO_RESULTADO),
    "arquivos_corpus": HASHES_RESULTADO,
    "ativado_no_aplicativo": False,
    "conferencia_editorial_integral": False,
}
with ARQUIVO_RELATORIO.open("x", encoding="utf-8", newline="\n") as arquivo:
    json.dump(RELATORIO_COLAB, arquivo, ensure_ascii=False, indent=2, allow_nan=False)
    arquivo.write("\n")
print("ZIP:", ARQUIVO_RESULTADO.name)
print("SHA-256:", RELATORIO_COLAB["arquivo_resultado_sha256"])
print("Relatório:", ARQUIVO_RELATORIO.name)


## Baixar os resultados

O navegador fará dois downloads: primeiro o ZIP, depois o relatório JSON.
Se ele bloquear downloads múltiplos, autorize-os no Colab ou use o painel
**Arquivos** à esquerda: encontre `volume19_execucoes`, abra a pasta desta
execução e escolha **Fazer download** nos dois arquivos.

Envie o JSON `volume19_vetores_colab_….json` para conferirmos a geração.
Guarde o ZIP: ele contém o novo volume 19 e não deve ser misturado manualmente
com a matriz antiga. Nada é importado automaticamente no Língua.


In [ ]:
from google.colab import files

if not ARQUIVO_RESULTADO.is_file() or not ARQUIVO_RELATORIO.is_file():
    raise RuntimeError("Faltam os arquivos finais. Termine a geração e a validação antes do download.")
if sha256_arquivo(ARQUIVO_RESULTADO) != RELATORIO_COLAB["arquivo_resultado_sha256"]:
    raise RuntimeError("O ZIP foi alterado desde a validação; o download foi interrompido.")
files.download(str(ARQUIVO_RESULTADO))
files.download(str(ARQUIVO_RELATORIO))
